# LeetCode #91 — Decode Ways: DP kiểu Fibonacci với điều kiện

**Mức độ:** Medium · **Chủ đề:** String, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def numDecodings(self, s: str) -> int:
```

Mục tiêu của notebook:

1. Định nghĩa `dp[i]` = số cách giải mã **tiền tố** `s[:i]`.
2. Hai lựa chọn ở mỗi bước: lấy **1 chữ số** (`1..9`) hoặc **2 chữ số** (`10..26`).
3. Xử lý số `0` — nguồn gốc của hầu hết các lỗi.
4. Tối ưu còn hai biến, giống Fibonacci / #70 Climbing Stairs.

# 1. Bài toán là gì?

Một thông điệp chữ hoa `A–Z` được mã hóa thành số theo bảng:

```text
'A' -> "1",  'B' -> "2",  ...,  'Z' -> "26"
```

Để **giải mã**, ta chia chuỗi số thành các nhóm, mỗi nhóm ứng với một chữ cái. Ví dụ `"11106"` có thể là:

- `"AAJF"` với cách chia `(1, 1, 10, 6)`
- `"KJF"` với cách chia `(11, 10, 6)`

Cách chia `(1, 11, 06)` **không hợp lệ**: `"06"` không phải mã của chữ nào (khác `"6"`).

Cho chuỗi `s` chỉ gồm chữ số, trả về **số cách giải mã**. Nếu không giải mã được, trả về `0`. Đáp án vừa với số nguyên 32-bit.

**Constraints (theo đề):**

- `1 <= s.length <= 100`
- `s` chỉ gồm chữ số và có thể có số `0` ở đầu.

---

## Ví dụ

```text
"12"   ->  2     "AB" (1 2) hoặc "L" (12)
"226"  ->  3     "BZ" (2 26), "VF" (22 6), "BBF" (2 2 6)
"06"   ->  0     "06" không hợp lệ, "0" cũng không
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)
2. Chuỗi (string) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [3]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [4]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [5]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [6]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.3 Áp dụng vào bài này

**Quy hoạch động** trên tiền tố: `dp[i]` = số cách giải mã `s[:i]`; ký tự cuối đứng một mình (nếu khác `'0'`) đóng góp `dp[i−1]`, hai ký tự cuối tạo số `10..26` đóng góp `dp[i−2]` — giống "leo cầu thang" nhưng có điều kiện.

# 3. Baseline: đệ quy thử mọi cách chia

`ways(i)` = số cách giải mã `s[i:]`:

- `s[i] == '0'` → 0 cách (không nhóm nào bắt đầu bằng `0`).
- Lấy 1 chữ số: `ways(i + 1)`.
- Lấy 2 chữ số nếu `10 <= int(s[i:i+2]) <= 26`: `ways(i + 2)`.

In [7]:
from typing import List
from functools import lru_cache
import random

def decode_naive(s: str):
    calls = 0
    def ways(i):
        nonlocal calls
        calls += 1
        if i == len(s):
            return 1
        if s[i] == "0":
            return 0
        total = ways(i + 1)
        if i + 1 < len(s) and 10 <= int(s[i:i + 2]) <= 26:
            total += ways(i + 2)
        return total
    return ways(0), calls

print(decode_naive("226"))
print(decode_naive("1" * 25))

(3, 7)


(121393, 317810)


`"111…1"` (25 chữ số) cần hơn 200 000 lời gọi — số lời gọi tăng như dãy Fibonacci. Có chỉ `n + 1` trạng thái `i` khác nhau → memo hoặc bảng.

# 4. Ý tưởng chính: bảng DP trên tiền tố

`dp[i]` = số cách giải mã `s[:i]` (tiền tố độ dài `i`).

```text
dp[0] = 1                     # chuỗi rỗng: một cách (không làm gì)
dp[i] = (dp[i-1] nếu s[i-1] != '0')               # chữ số cuối đứng một mình
      + (dp[i-2] nếu i >= 2 và 10 <= int(s[i-2:i]) <= 26)   # hai chữ số cuối thành một nhóm
```

`dp[0] = 1` là "cách rỗng" — cần để `dp[1]`, `dp[2]` tính đúng.

In [8]:
def decode_table(s: str) -> int:
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        if s[i - 1] != "0":
            dp[i] += dp[i - 1]
        if i >= 2 and 10 <= int(s[i - 2:i]) <= 26:
            dp[i] += dp[i - 2]
    return dp[n]

for s in ["12", "226", "06", "11106", "10", "2101"]:
    print(f"{s!r:>8} -> {decode_table(s)}")

    '12' -> 2
   '226' -> 3
    '06' -> 0
 '11106' -> 2
    '10' -> 1
  '2101' -> 1


# 5. Mô phỏng

In [9]:
def trace_decode(s: str) -> int:
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    print(f"s = {s!r}")
    print(f"{'i':>2} {'1 chữ số':>12} {'2 chữ số':>12}  dp[i]")
    for i in range(1, n + 1):
        one = s[i - 1]
        two = s[i - 2:i] if i >= 2 else ""
        a = dp[i - 1] if one != "0" else 0
        b = dp[i - 2] if two and 10 <= int(two) <= 26 else 0
        dp[i] = a + b
        one_txt = f"'{one}'→+{a}" if one != "0" else f"'{one}'→ 0"
        two_txt = (f"'{two}'→+{b}" if b or (two and 10 <= int(two) <= 26) else f"'{two}'→ 0") if two else "-"
        print(f"{i:>2} {one_txt:>12} {two_txt:>12}  {dp[i]}")
    return dp[n]

_ = trace_decode("2101")

s = '2101'
 i     1 chữ số     2 chữ số  dp[i]
 1       '2'→+1            -  1
 2       '1'→+1      '21'→+1  2
 3       '0'→ 0      '10'→+1  1
 4       '1'→+1      '01'→ 0  1


Với `"2101"`:

| i | Ghi chú |
|---|---|
| 2 | `'1'` đứng một mình được; `'21'` hợp lệ → `dp[2] = dp[1] + dp[0] = 2` |
| 3 | `'0'` không đứng một mình được; `'10'` hợp lệ → `dp[3] = dp[1] = 1` (bắt buộc ghép `10`) |
| 4 | `'1'` được; `'01'` **không** hợp lệ (bắt đầu bằng 0) → `dp[4] = dp[3] = 1` |

Chỉ có một cách: `2 10 1` = `"BJA"`.

# 6. Tối ưu bộ nhớ: hai biến

`dp[i]` chỉ cần `dp[i-1]` và `dp[i-2]` → giữ `prev2`, `prev1` như Fibonacci.

In [10]:
def decode_two_vars(s: str) -> int:
    prev2, prev1 = 1, 1 if s[0] != "0" else 0       # dp[0], dp[1]
    for i in range(2, len(s) + 1):
        cur = 0
        if s[i - 1] != "0":
            cur += prev1
        if 10 <= int(s[i - 2:i]) <= 26:
            cur += prev2
        prev2, prev1 = prev1, cur
    return prev1

print([decode_two_vars(s) for s in ["12", "226", "06", "11106", "10", "2101"]])

[2, 3, 0, 2, 1, 1]


# 7. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def numDecodings(self, s: str) -> int:
        prev2, prev1 = 1, 1 if s[0] != "0" else 0
        for i in range(2, len(s) + 1):
            cur = 0
            if s[i - 1] != "0":
                cur += prev1
            two = int(s[i - 2:i])
            if 10 <= two <= 26:
                cur += prev2
            prev2, prev1 = prev1, cur
        return prev1

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


def brute_decode(s: str) -> int:
    if not s:
        return 1
    total = 0
    for L in (1, 2):
        if len(s) >= L and s[0] != "0" and 1 <= int(s[:L]) <= 26:
            total += brute_decode(s[L:])
    return total


sol = Solution()
check("Ví dụ 1", sol.numDecodings("12"), 2)
check("Ví dụ 2", sol.numDecodings("226"), 3)
check("Ví dụ 3", sol.numDecodings("06"), 0)
check("11106", sol.numDecodings("11106"), 2)
check("'0' đơn lẻ", sol.numDecodings("0"), 0)
check("'10'", sol.numDecodings("10"), 1)
check("'100' (00 kẹt)", sol.numDecodings("100"), 0)
check("'27' (> 26)", sol.numDecodings("27"), 1)
check("'30' (30 > 26)", sol.numDecodings("30"), 0)
check("100 số 1", sol.numDecodings("1" * 100), 573147844013817084101)

random.seed(91)
for _ in range(5000):
    s = "".join(random.choice("0123456789" if random.random() < 0.3 else "0122") for _ in range(random.randint(1, 14)))
    assert sol.numDecodings(s) == brute_decode(s) == decode_table(s), s
print("5000 chuỗi ngẫu nhiên: khớp brute force")

Ví dụ 1                PASS  2
Ví dụ 2                PASS  3
Ví dụ 3                PASS  0
11106                  PASS  2
'0' đơn lẻ             PASS  0
'10'                   PASS  1
'100' (00 kẹt)         PASS  0
'27' (> 26)            PASS  1
'30' (30 > 26)         PASS  0
100 số 1               PASS  573147844013817084101


5000 chuỗi ngẫu nhiên: khớp brute force


(Đề nói đáp án vừa 32-bit cho **test của LeetCode**; với `"1" * 100` số cách là số Fibonacci thứ 101 — rất lớn, Python vẫn tính chính xác nhờ số nguyên không giới hạn.)

# 8. Vì sao đúng? Độ phức tạp?

Mỗi cách giải mã `s[:i]` kết thúc bằng một nhóm cuối có độ dài 1 hoặc 2 — hai trường hợp **rời nhau**:

- Nhóm cuối là `s[i-1]` (hợp lệ ⇔ khác `'0'`): phần trước là một cách giải mã `s[:i-1]` → `dp[i-1]` cách.
- Nhóm cuối là `s[i-2:i]` (hợp lệ ⇔ trong `10..26`, tự động loại `"0x"`): phần trước có `dp[i-2]` cách.

Cộng lại được `dp[i]`.

```text
Time:  O(n)
Space: O(1) (hai biến) | O(n) (bảng)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Kiểm tra hai chữ số bằng `<= 26` mà quên `>= 10`

`"06"` → `int("06") = 6 <= 26` → bị coi là hợp lệ.

In [13]:
def wrong_no_lower_bound(s: str) -> int:
    n = len(s)
    dp = [1] + [0] * n
    for i in range(1, n + 1):
        if s[i - 1] != "0":
            dp[i] += dp[i - 1]
        if i >= 2 and int(s[i - 2:i]) <= 26:          # SAI: thiếu >= 10
            dp[i] += dp[i - 2]
    return dp[n]

for t in ["06", "101", "2101"]:
    print(f"{t!r:>7}  Sai: {wrong_no_lower_bound(t)} | Đúng: {sol.numDecodings(t)}")

   '06'  Sai: 1 | Đúng: 0
  '101'  Sai: 2 | Đúng: 1
 '2101'  Sai: 3 | Đúng: 1


## Lỗi 2: Coi `'0'` là một chữ số đứng một mình được

In [14]:
def wrong_zero_single(s: str) -> int:
    n = len(s)
    dp = [1] + [0] * n
    for i in range(1, n + 1):
        dp[i] += dp[i - 1]                            # SAI: kể cả khi s[i-1] == '0'
        if i >= 2 and 10 <= int(s[i - 2:i]) <= 26:
            dp[i] += dp[i - 2]
    return dp[n]

for t in ["10", "0", "100"]:
    print(f"{t!r:>6}  Sai: {wrong_zero_single(t)} | Đúng: {sol.numDecodings(t)}")

  '10'  Sai: 2 | Đúng: 1
   '0'  Sai: 1 | Đúng: 0
 '100'  Sai: 2 | Đúng: 0


## Lỗi 3: So sánh chuỗi thay vì số

`"3" <= "26"`? So sánh chuỗi theo thứ tự từ điển: `"3" > "26"` vì `'3' > '2'`. Và `"9" > "26"`, nhưng `"100" < "26"`. Luôn đổi `int(...)` hoặc so sánh cẩn thận từng ký tự.

In [15]:
print('"27" <= "26":', "27" <= "26", '| "3" <= "26":', "3" <= "26", '| "100" <= "26":', "100" <= "26")

"27" <= "26": False | "3" <= "26": False | "100" <= "26": True


## Lỗi 4: `dp[0] = 0`

Khi đó mọi giá trị đều bằng 0. `dp[0] = 1` biểu diễn "một cách giải mã chuỗi rỗng".

# 10. Mẹo nhận diện pattern

**DP trên tiền tố, mỗi bước "ăn" 1 hoặc 2 phần tử** → dạng Fibonacci có điều kiện:

| Bài | Bước 1 | Bước 2 |
|---|---|---|
| #70 Climbing Stairs | luôn được | luôn được |
| #91 Decode Ways | `s[i-1] != '0'` | `10 <= s[i-2:i] <= 26` |
| #746 Min Cost Climbing Stairs | `min` thay cho `+` | |
| #639 Decode Ways II (Hard) | có ký tự `*` | nhân số lựa chọn |

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng `dp` cho `"1201234"`.

## Bài 2 — Điền chỗ trống (bảng DP)

In [16]:
def exercise_2_num_decodings(s: str) -> int:
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        # TODO: cộng dp[i-1] nếu chữ số cuối hợp lệ; cộng dp[i-2] nếu hai chữ số cuối hợp lệ
        pass
    return dp[n]


# assert exercise_2_num_decodings("226") == 3
# assert exercise_2_num_decodings("06") == 0
# assert exercise_2_num_decodings("11106") == 2

## Bài 3 — Liệt kê mọi cách giải mã

Trả về list các chuỗi chữ cái (thứ tự tùy ý).

```python
all_decodings("226") -> ["BBF", "BZ", "VF"]
```

In [17]:
def exercise_3_all_decodings(s: str) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_3_all_decodings("226")) == ["BBF", "BZ", "VF"]
# assert exercise_3_all_decodings("06") == []
# assert sorted(exercise_3_all_decodings("11106")) == ["AAJF", "KJF"]

## Bài 4 — Biến thể: bảng mã khác

Giả sử bảng mã là `'A' -> "1"`, …, `'Z' -> "26"` **và** thêm các ký hiệu `27..35` (tổng 35 ký hiệu). Đếm số cách giải mã. Tổng quát: nhóm 2 chữ số hợp lệ nếu trong `[10, max_code]`.

```python
num_decodings_max("27", 35) -> 2
num_decodings_max("27", 26) -> 1
```

In [18]:
def exercise_4_num_decodings_max(s: str, max_code: int) -> int:
    # TODO
    pass


# assert exercise_4_num_decodings_max("27", 35) == 2
# assert exercise_4_num_decodings_max("27", 26) == 1
# assert exercise_4_num_decodings_max("226", 26) == 3

## Bài 5 — Mini challenge: Decode Ways II (LeetCode #639, Hard)

Chuỗi có thêm ký tự `'*'` đại diện cho **bất kỳ** chữ số `1..9` (không phải 0). Đếm số cách, lấy **modulo `10^9 + 7`**.

```python
num_decodings_star("*")   -> 9
num_decodings_star("1*")  -> 18     # 9 cách "1 x" + 9 cách "1x"
num_decodings_star("2*")  -> 15     # 9 + 6 (21..26)
```

Gợi ý nhanh: đếm số giá trị của một ký tự đơn (`'*'` → 9, `'0'` → 0, khác → 1) và của một cặp (xét các trường hợp `**`, `*d`, `d*`, `dd`).

In [19]:
def exercise_5_num_decodings_star(s: str) -> int:
    # TODO
    pass


# assert exercise_5_num_decodings_star("*") == 9
# assert exercise_5_num_decodings_star("1*") == 18
# assert exercise_5_num_decodings_star("2*") == 15
# assert exercise_5_num_decodings_star("**") == 96

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_decode` sau khi tự làm.

**Bài 3**
- Mức 1: backtracking `go(i, path)`: lấy 1 hoặc 2 chữ số hợp lệ, `chr(ord('A') + code - 1)`.

**Bài 4**
- Mức 1: thay `26` bằng `max_code` trong điều kiện hai chữ số.

**Bài 5**
- Mức 1: `one(c)`: `'*'` → 9, `'0'` → 0, khác → 1.
- Mức 2: `two(a, b)`: `**` → 15 (11–19 và 21–26); `*d` → 2 nếu `d <= 6` (1d, 2d) ngược lại 1; `1*` → 9; `2*` → 6; `dd` → 1 nếu `10..26`; còn lại 0.
- Mức 3: `dp[i] = one(s[i-1]) · dp[i-1] + two(s[i-2], s[i-1]) · dp[i-2]` (mod).

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [20]:
_ = trace_decode("1201234")

s = '1201234'
 i     1 chữ số     2 chữ số  dp[i]
 1       '1'→+1            -  1
 2       '2'→+1      '12'→+1  2
 3       '0'→ 0      '20'→+1  1
 4       '1'→+1      '01'→ 0  1
 5       '2'→+1      '12'→+1  2
 6       '3'→+2      '23'→+1  3
 7       '4'→+3      '34'→ 0  3


## Đáp án Bài 2

In [21]:
def answer_2_num_decodings(s: str) -> int:
    return decode_table(s)

assert answer_2_num_decodings("226") == 3
assert answer_2_num_decodings("06") == 0
assert answer_2_num_decodings("11106") == 2
print("Bài 2 PASS (giống decode_table ở mục 4)")

Bài 2 PASS (giống decode_table ở mục 4)


## Đáp án Bài 3

In [22]:
def answer_3_all_decodings(s: str) -> List[str]:
    result = []
    def go(i, path):
        if i == len(s):
            result.append("".join(path))
            return
        for L in (1, 2):
            part = s[i:i + L]
            if len(part) == L and part[0] != "0" and 1 <= int(part) <= 26:
                path.append(chr(ord("A") + int(part) - 1))
                go(i + L, path)
                path.pop()
    go(0, [])
    return result

assert sorted(answer_3_all_decodings("226")) == ["BBF", "BZ", "VF"]
assert answer_3_all_decodings("06") == []
assert sorted(answer_3_all_decodings("11106")) == ["AAJF", "KJF"]
for _ in range(1000):
    s = "".join(random.choice("01226") for _ in range(random.randint(1, 10)))
    assert len(answer_3_all_decodings(s)) == sol.numDecodings(s)
print("Bài 3 PASS (số cách liệt kê khớp numDecodings)")

Bài 3 PASS (số cách liệt kê khớp numDecodings)


## Đáp án Bài 4

In [23]:
def answer_4_num_decodings_max(s: str, max_code: int) -> int:
    n = len(s)
    dp = [1] + [0] * n
    for i in range(1, n + 1):
        if s[i - 1] != "0":
            dp[i] += dp[i - 1]
        if i >= 2 and 10 <= int(s[i - 2:i]) <= max_code:
            dp[i] += dp[i - 2]
    return dp[n]

assert answer_4_num_decodings_max("27", 35) == 2
assert answer_4_num_decodings_max("27", 26) == 1
assert answer_4_num_decodings_max("226", 26) == 3
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [24]:
def answer_5_num_decodings_star(s: str) -> int:
    MOD = 10**9 + 7

    def one(c):
        if c == "*":
            return 9
        return 0 if c == "0" else 1

    def two(a, b):
        if a == "*" and b == "*":
            return 15
        if a == "*":
            return 2 if b <= "6" else 1
        if b == "*":
            return 9 if a == "1" else (6 if a == "2" else 0)
        return 1 if 10 <= int(a + b) <= 26 else 0

    prev2, prev1 = 1, one(s[0])
    for i in range(1, len(s)):
        cur = (one(s[i]) * prev1 + two(s[i - 1], s[i]) * prev2) % MOD
        prev2, prev1 = prev1, cur
    return prev1

assert answer_5_num_decodings_star("*") == 9
assert answer_5_num_decodings_star("1*") == 18
assert answer_5_num_decodings_star("2*") == 15
assert answer_5_num_decodings_star("**") == 96

import itertools
def brute_star(s):
    stars = [i for i, c in enumerate(s) if c == "*"]
    total = 0
    for digits in itertools.product("123456789", repeat=len(stars)):
        t = list(s)
        for i, d in zip(stars, digits):
            t[i] = d
        total += sol.numDecodings("".join(t))
    return total

for _ in range(500):
    s = "".join(random.choice("*0126*7") for _ in range(random.randint(1, 5)))
    assert answer_5_num_decodings_star(s) == brute_star(s) % (10**9 + 7), s
print("Bài 5 PASS (khớp brute force thay '*' bằng mọi chữ số)")

Bài 5 PASS (khớp brute force thay '*' bằng mọi chữ số)

Chú ý: so sánh `b <= "6"` giữa hai **ký tự đơn** là an toàn (vì `'0' < '1' < … < '9'`), khác với Lỗi 3 khi so chuỗi nhiều ký tự.

# 14. Tổng kết

> `dp[i] = [s[i-1] != '0'] · dp[i-1] + [10 <= s[i-2:i] <= 26] · dp[i-2]`, `dp[0] = 1`. Giống Fibonacci nhưng mỗi bước có điều kiện. Số `0` chỉ hợp lệ khi là chữ số thứ hai của `10` hoặc `20`.

---

## Checklist trước khi nộp

- [ ] `dp[0] = 1`?
- [ ] Một chữ số: khác `'0'`?
- [ ] Hai chữ số: `10 <= int(...) <= 26` (có cận dưới)?
- [ ] So sánh số, không so sánh chuỗi nhiều ký tự?